# Что случилось с выручкой после привязки категорий?

У аналитика четыре оплаченные строки и отдельный справочник товаров. Нужно добавить категорию, удержать в отчёте товар без описания и проверить сумму до и после соединения.

Строки продаж заданы прямо в первой ячейке. Товар B встречается дважды, P и X по одному разу. Справочник знает коды B и P. Суммы даны в тенге.

Выполняйте ячейки по порядку. Перед запуском попробуйте предсказать вывод.

## Шаг 1. Каков контрольный итог до соединения?

In [ ]:
import pandas as pd

sales = pd.DataFrame({
    "sku": ["B", "P", "B", "X"],
    "revenue_kzt": [1000, 600, 500, 300],
})
catalog = pd.DataFrame({"sku": ["B", "P"], "category": ["Бумага", "Письмо"]})
print(len(sales), sales["revenue_kzt"].sum())

**Обсуждение.** Четыре строки дают 2 400 ₸. Код B законно встречается в продажах дважды. В справочнике каждому коду соответствует одна запись.

## Шаг 2. Как увидеть товар, которому не нашли описание?

In [ ]:
joined = sales.merge(catalog, on="sku", how="left",
                     validate="many_to_one", indicator=True)
unmatched = joined.loc[joined["_merge"].eq("left_only"), "sku"]
print(unmatched.tolist())
print(len(joined), joined["revenue_kzt"].sum())

**Обсуждение.** Левое соединение оставило все продажи. Индикатор показывает, где отсутствует совпадение справа. Проверка many_to_one потребует уникальных ключей в справочнике.

## Шаг 3. Как включить неизвестную категорию в итог?

In [ ]:
joined["category"] = joined["category"].fillna("Без категории")
totals = joined.groupby("category")["revenue_kzt"].sum()
print(totals.to_dict())
assert totals.sum() == sales["revenue_kzt"].sum()

**Обсуждение.** Здесь пропуск означает неизвестную категорию. Мы присвоили ей явное имя и сохранили сумму 300 ₸ в отдельной строке. Денежные значения остались исходными.

## Поиск логической ошибки
Почему результат может ввести в заблуждение?

In [ ]:
inner = sales.merge(catalog, on="sku", how="inner", validate="many_to_one")
print(len(inner), inner["revenue_kzt"].sum())

Внутреннее соединение исключило товар X. Три оставшиеся строки арифметически корректны, но теперь в отчёте потеряны 300 ₸.

## Измените задачу
Добавьте код X с категорией «Бумага» в справочник и повторите левое соединение с исходными продажами. Сравните выручку по категориям и общий итог.

In [ ]:
# Ваше решение

Для проверки: Пустых соответствий больше нет. «Бумага» даёт 1 800 ₸, «Письмо» — 600 ₸. Всего по-прежнему четыре строки и 2 400 ₸.